In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
device = "cuda"
model_name = "Qwen/Qwen2.5-Coder-1.5B" 

/root/workspace/env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from typing import Any


tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True, padding_side="right")
# tokenizer.pad_token = tokenizer.eos_token = "<|endoftext|>"
# --- Dataset ---
dataset = load_dataset("json", data_files="data/train_data_repo_gaussian_window.jsonl")
def tokenize(example):
    return tokenizer(example["text"], truncation=False, padding=False, max_length=2048)
dataset = dataset.map(tokenize, batched=True, remove_columns=["text"])
# Used to pad tokens, since each sample have different length
class CustomCausalCollator:
    def __init__(self, tokenizer) -> None:
        self.tokenizer = tokenizer
        self.mid_token = 151660
    def __call__(self, features) -> Any:
        batch = self.tokenizer.pad(
            features,
            return_tensors="pt"
        )
        input_ids = batch["input_ids"]
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        for i in range(input_ids.size(0)):
            row = input_ids[i]
            pos = (row == self.mid_token).nonzero(as_tuple=True)
            if pos[0].numel() > 0:
                mid_idx = pos[0].item()
                labels[i, :mid_idx+1] = -100 # Mask <|fim_middle|> too                
        batch["labels"] = labels
        return batch
data_collator = CustomCausalCollator(tokenizer)

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
).to(device, dtype=torch.bfloat16)

model.gradient_checkpointing_enable()
model.config.use_cache = False

# --- LoRA config ---
lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_cfg).to(device, dtype=torch.bfloat16)

# --- Training ---
args = TrainingArguments(
    output_dir="qwen25-15-c-w-ft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    num_train_epochs=1,
    bf16=True,               # enables CUDA bfloat16
    logging_steps=128,
    save_strategy="steps",
    save_steps=128, # 4096 samples
    save_total_limit=100,      
    save_safetensors=True,   # ✅ use safetensors format (recommended)
    optim="adamw_torch",
    report_to="none",
)

trainer = Trainer(model=model, args=args, train_dataset=dataset["train"], data_collator=data_collator)
trainer.train()

`torch_dtype` is deprecated! Use `dtype` instead!


You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Step,Training Loss
128,0.987400
256,0.897000
384,0.865800
512,0.850700


TrainOutput(global_step=638, training_loss=0.889879352246706, metrics={'train_runtime': 4385.5969, 'train_samples_per_second': 1.162, 'train_steps_per_second': 0.145, 'total_flos': 7.026434743282483e+16, 'train_loss': 0.889879352246706, 'epoch': 1.0})